# 3. CamemBERT : un Transformer qui comprend le contexte

**Objectif** : fine-tuner CamemBERT (un modèle pré-entraîné sur du français) pour classer les avis Allociné, puis le comparer à TF-IDF (93,8 %).

### Avant de commencer (dans Google Colab)
1. **Exécution → Modifier le type d'exécution → GPU T4**, puis Enregistrer. Sans GPU, l'entraînement prendrait des heures.
2. Exécute les cellules **dans l'ordre** avec Maj + Entrée.
3. Garde l'onglet ouvert pendant l'entraînement : Colab peut couper une session inactive, et tout ce qui est en mémoire est alors perdu.

In [19]:
# Installe les bibliothèques (déjà présentes sur Colab en général, ça ne coûte rien de vérifier)
!pip install -q transformers datasets accelerate sentencepiece

In [20]:
import torch

print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Carte :", torch.cuda.get_device_name(0))
else:
    print("⚠️ Pas de GPU : active-le dans Exécution > Modifier le type d'exécution")

GPU disponible : True
Carte : Tesla T4


## 3.1 Charger les données

On les télécharge directement depuis Hugging Face : pas besoin d'envoyer tes fichiers sur Colab.

**On n'entraîne que sur 20 000 avis** (au lieu de 160 000) : sur un GPU gratuit, ça prend environ 5 à 10 minutes au lieu d'une heure. On évalue sur **toute la validation** (20 000 avis), la même que pour TF-IDF, pour que la comparaison soit juste.

Remarque : TF-IDF a été entraîné sur 160 000 avis, CamemBERT n'en verra que 20 000, soit **8 fois moins**. Si CamemBERT fait quand même mieux, c'est grâce à ce qu'il a appris avant (le pré-entraînement).

In [21]:
from datasets import load_dataset

N_TRAIN = 20_000

ds = load_dataset("tblard/allocine")
train_petit = ds["train"].shuffle(seed=42).select(range(N_TRAIN))
val = ds["validation"]

print(train_petit)
print(val)

Dataset({
    features: ['review', 'label'],
    num_rows: 20000
})
Dataset({
    features: ['review', 'label'],
    num_rows: 20000
})


## 3.2 La tokenisation en sous-mots

TF-IDF découpait le texte en **mots entiers**. CamemBERT découpe en **sous-mots** (*tokens*) :
- les mots fréquents restent entiers (`film`) ;
- les mots rares sont coupés en morceaux connus (`ininregardable` → `▁in` + `in` + `regard` + `able`).

**Pourquoi ?** Le vocabulaire est limité (32 000 tokens), mais avec des morceaux on peut écrire **n'importe quel mot**, même une faute de frappe ou un mot jamais vu. Et le modèle peut deviner le sens d'un mot rare grâce à ses morceaux (`in-` = négation).

Le symbole `▁` indique un début de mot. Chaque token a un **numéro** (son identifiant dans le vocabulaire) : c'est cette liste de numéros que lit le modèle.

In [4]:
from transformers import AutoTokenizer

NOM_MODELE = "almanach/camembert-base"
tokenizer = AutoTokenizer.from_pretrained(NOM_MODELE)

phrase = "Un navet ininregardable, mais pas mauvais du tout."
print(tokenizer.tokenize(phrase))
print(tokenizer(phrase)["input_ids"])

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

['▁Un', '▁na', 'vet', '▁in', 'in', 'regard', 'able', ',', '▁mais', '▁pas', '▁mauvais', '▁du', '▁tout', '.']
[5, 153, 3650, 6957, 378, 236, 25494, 651, 7, 65, 34, 1973, 25, 66, 9, 6]


### À TOI (3.2)
1. Tokenise ton **avis piège n°2** (« Film pas mauvais, qui se laisse regarder… ») avec `tokenizer.tokenize(...)`.
2. Compare le **nombre de tokens** (`len(...)`) au **nombre de mots** (`len(texte.split())`). Pourquoi y a-t-il plus de tokens que de mots ?
3. La liste `input_ids` commence et finit par deux numéros spéciaux. Affiche les tokens correspondants avec `tokenizer.convert_ids_to_tokens(ids)`. Que sont `<s>` et `</s>` à ton avis ?

In [5]:
# À TOI
pieges_2= 'Film pas mauvais, qui se laisse regarder. Très belle performance de Wahlberg pas habitué à jouer ce genre de rôle et qui réussit son pari.'
tok=tokenizer.tokenize(pieges_2)
ids = tokenizer(pieges_2)["input_ids"]
print(tok)
print(len(tok))
print(pieges_2.split())
print(len(pieges_2.split()))
print(tokenizer.convert_ids_to_tokens(ids))


['▁Film', '▁pas', '▁mauvais', ',', '▁qui', '▁se', '▁laisse', '▁regarder', '.', '▁Très', '▁belle', '▁performance', '▁de', '▁Wa', 'hl', 'berg', '▁pas', '▁habitué', '▁à', '▁jouer', '▁ce', '▁genre', '▁de', '▁rôle', '▁et', '▁qui', '▁réussit', '▁son', '▁pari', '.']
30
['Film', 'pas', 'mauvais,', 'qui', 'se', 'laisse', 'regarder.', 'Très', 'belle', 'performance', 'de', 'Wahlberg', 'pas', 'habitué', 'à', 'jouer', 'ce', 'genre', 'de', 'rôle', 'et', 'qui', 'réussit', 'son', 'pari.']
25
['<s>', '▁Film', '▁pas', '▁mauvais', ',', '▁qui', '▁se', '▁laisse', '▁regarder', '.', '▁Très', '▁belle', '▁performance', '▁de', '▁Wa', 'hl', 'berg', '▁pas', '▁habitué', '▁à', '▁jouer', '▁ce', '▁genre', '▁de', '▁rôle', '▁et', '▁qui', '▁réussit', '▁son', '▁pari', '.', '</s>']


Mes réponses (3.2) :

1. Plus de tokens que de mots : on a 30 tokens pour 25 mots. `split()` coupe seulement sur les espaces, alors que CamemBERT découpe en sous-mots. Il y a deux raisons :
. la ponctuation devient un token à part : mauvais, → ▁mauvais + ,, ce qui fait +3 avec , . . ;
. les mots rares sont découpés en morceaux connus du vocabulaire : Wahlberg → ▁Wa + hl + berg (+2).

Les mots fréquents (habitué, performance, réussit) restent entiers. Le ▁ marque le début d’un mot.

2. `<s>` et `</s>` sont des tokens spéciaux ajoutés automatiquement : `<s>` marque le début du texte, `</s>` la fin. Ils n’apparaissent pas avec tokenize(), mais ils sont bien dans input_ids (32 au lieu de 30). Grâce à l’attention, `<s>` regarde tous les autres tokens et contient un résumé de l’avis : c’est son vecteur que la couche de décision utilise pour classer l’avis en positif ou négatif.

## 3.3 Tokeniser tout le jeu de données

CamemBERT ne peut lire que **512 tokens au maximum**. On coupe à **256** (`truncation=True, max_length=256`) : c'est environ deux fois plus rapide. Mais une partie des avis sera coupée : tu vas mesurer combien juste après.

`.map(..., batched=True)` applique la fonction à tous les avis, par paquets, très rapidement.

In [42]:
def tokeniser(batch):
    return tokenizer(batch["review"], truncation=True, max_length=256)

train_tok = train_petit.map(tokeniser, batched=True)
val_tok = val.map(tokeniser, batched=True)
train_tok

Dataset({
    features: ['review', 'label', 'input_ids', 'attention_mask'],
    num_rows: 20000
})

### À TOI (3.3)
Vérifie notre choix de 256 : **quel pourcentage des avis de validation dépasse 256 tokens** (et sera donc coupé) ?

*Indice* : tokenise les 2 000 premiers avis de `val["review"]` **sans** `truncation`, calcule la longueur de chaque liste `input_ids`, puis la proportion supérieure à 256. (Rappel : `(liste_de_longueurs > 256).mean()` marche si tu mets les longueurs dans un tableau NumPy ou une Series pandas.)

In [31]:
# À TOI
import numpy as np

resultat=tokenizer(val["review"][:2000])
longueurs=np.array([len(i) for i in resultat["input_ids"]])
print(len(longueurs[longueurs>256])/len(longueurs)*100,"%")


14.75 %


**Question** : la troncature garde le **début** de l'avis et coupe la **fin**. Pense à ton avis piège n°1 (« … Verdict : c'est sympa… mais très lourd »). Pourquoi couper la fin d'une critique peut-il être un problème ?

**Ma réponse (3.3)** : environ 15 % des avis dépassent 256 tokens et seront coupés. C’est un risque, car beaucoup de critiques citent d’abord les points positifs puis les points négatifs, et donnent leur verdict final à la fin. Si la troncature supprime la fin, le modèle ne voit que le début, qui peut être trompeur : mon avis piège n°1 fait 413 tokens, il est donc coupé ; ici le verdict (« très lourd », token 160) est conservé, mais si l’auteur l’avait écrit dans les dernières lignes, le modèle ne l’aurait jamais vu. Pour limiter ce problème, on pourrait passer à max_length=512 (plus lent), ou garder le début et la fin de l’avis.

## 3.4 Le modèle

`AutoModelForSequenceClassification` charge CamemBERT **et ajoute au-dessus une petite couche de décision** à 2 sorties (négatif, positif).

- Le corps de CamemBERT (110 millions de paramètres) est **déjà entraîné** : il comprend le français.
- La couche de décision est **neuve**, avec des poids au hasard : c'est elle qui va surtout apprendre.

Un message d'avertissement va dire que certains poids sont « newly initialized » : **c'est normal**, c'est justement cette nouvelle couche.

In [32]:
from transformers import AutoModelForSequenceClassification

modele = AutoModelForSequenceClassification.from_pretrained(NOM_MODELE, num_labels=2)
print(f"{modele.num_parameters():,} paramètres")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: almanach/camembert-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


110,623,490 paramètres


## 3.5 L'entraînement (fine-tuning)

On utilise le `Trainer` de Hugging Face, qui gère la boucle d'entraînement pour nous. Les réglages importants :

| Paramètre | Valeur | Signification |
|---|---|---|
| `num_train_epochs` | 1 | le modèle voit chaque avis d'entraînement **une fois** |
| `per_device_train_batch_size` | 16 | il apprend par paquets de 16 avis |
| `learning_rate` | 2e-5 | **très petit** : on ajuste doucement un modèle qui sait déjà beaucoup, sans lui faire « oublier » le français |
| `fp16` | True | calculs en demi-précision : environ 2 fois plus rapide sur GPU |

Le **`DataCollatorWithPadding`** complète les avis courts avec des tokens vides pour que tous les avis d'un paquet aient la même longueur (un GPU calcule sur des tableaux rectangulaires).

`calculer_accuracy` dit au Trainer comment noter le modèle : le modèle sort deux scores par avis (les *logits*), on prend le plus grand avec `argmax`, et on compare aux vraies réponses.

⏱️ **Durée attendue : 5 à 10 minutes** sur un GPU T4. La barre de progression s'affiche pendant l'entraînement.

In [33]:
import numpy as np
from sklearn.metrics import accuracy_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments


def calculer_accuracy(eval_pred):
    logits, vraies_reponses = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(vraies_reponses, predictions)}


reglages = TrainingArguments(
    output_dir="camembert-allocine",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    eval_strategy="no",      # on évaluera nous-mêmes après l'entraînement
    save_strategy="no",      # pas de sauvegarde intermédiaire (on sauvegarde à la fin)
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=modele,
    args=reglages,
    train_dataset=train_tok,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=calculer_accuracy,
)

trainer.train()

Step,Training Loss
100,0.443723
200,0.196245
300,0.200282
400,0.166249
500,0.174329
600,0.149701
700,0.158908
800,0.146396
900,0.169087
1000,0.148107


TrainOutput(global_step=1250, training_loss=0.1857753532409668, metrics={'train_runtime': 255.6886, 'train_samples_per_second': 78.22, 'train_steps_per_second': 4.889, 'total_flos': 2591265673153920.0, 'train_loss': 0.1857753532409668, 'epoch': 1.0})

## 3.6 À TOI : évaluer sur la validation

1. Utilise `trainer.evaluate(val_tok)` pour obtenir l'accuracy sur les 20 000 avis de validation (≈ 1 à 2 minutes).
2. Complète le tableau de comparaison ci-dessous.
3. Calcule la **réduction d'erreurs** par rapport à TF-IDF + bigrammes. Rappel : erreur = 1 − accuracy. Par exemple, passer de 6 % à 3 % d'erreurs, c'est 50 % d'erreurs en moins.

In [44]:
# À TOI
trainer.evaluate(val_tok)


Training Loss,Validation Loss,Step,Accuracy
0.139895,0.140119,1250,0.958050


{'eval_loss': 0.1401188224554062, 'eval_accuracy': 0.95805}

| Modèle | Données d'entraînement | Accuracy (validation) |
|---|---|---|
| TF-IDF + régression logistique | 160 000 avis | 92,5 % |
| TF-IDF + bigrammes | 160 000 avis | 93,8 % |
| **CamemBERT** | 20 000 avis | **95,8 %** |

**Réduction d'erreurs par rapport aux bigrammes** : erreurs 6,2 % → 4,2 %, soit (6,2 − 4,2) / 6,2 ≈ **32 % d'erreurs en moins**

**Ma conclusion** : CamemBERT atteint 95,8 % contre 93,8 % pour TF-IDF bigrammes, soit environ 32 % d'erreurs en moins, alors qu'il a été entraîné sur 8 fois moins d'avis (20 000 contre 160 000) et en 4 minutes. C'est l'effet du pré-entraînement : il connaissait déjà le français avant de voir nos avis. En contrepartie, il est beaucoup plus lourd (110 millions de paramètres, GPU nécessaire pour l'entraîner) et plus lent à prédire qu'une régression logistique.

## 3.7 À TOI : tes avis pièges

Écris une fonction `predire(texte)` qui renvoie la probabilité que l'avis soit positif.

*Les étapes* :
1. `entrees = tokenizer([texte], return_tensors="pt", truncation=True, max_length=256).to(modele.device)` : tokenise et envoie sur le GPU ;
2. `with torch.no_grad():` puis `logits = modele(**entrees).logits` : `no_grad` dit à PyTorch qu'on ne s'entraîne pas, ce qui économise de la mémoire ;
3. `probas = torch.softmax(logits, dim=-1)` : le **softmax** transforme les 2 scores en 2 probabilités qui somment à 1. C'est la généralisation de la sigmoïde de la régression logistique ;
4. renvoie `probas[0, 1].item()`, la probabilité de « positif ».

Puis teste tes **3 avis pièges** (« génial » qui parle d'un autre film, « Film pas mauvais… », et ton avis inventé « Un film d'un grand intérêt, pas mauvais du tout. »).

**Question** : CamemBERT fait-il mieux que TF-IDF sur ces avis ? Pourquoi, d'après ce que tu sais de l'attention ?

In [49]:
# À TOI
def predire(texte):
    entrees = tokenizer([texte], return_tensors="pt", truncation=True, max_length=256).to(modele.device)
    with torch.no_grad():
        logits = modele(**entrees).logits
    probas = torch.softmax(logits, dim=-1)
    return probas[0, 1].item()
print(predire("Après avoir réussi à mêler film horreur et teen-movie avec le génial Destination finale, le réalisateur James Wong s'est embarqué sous les sirènes d'Hollywood dans un film de science-fiction bourré d'action et d'effets spéciaux. Sur le papelard, The One avait l'air plutôt intéressant avec cette histoire de combat entre personnages de plusieurs dimensions parallèles. Et puis voir Jet Li dans un double rôle était une idée alléchante pour les fans de la star asiatique. En somme, The One promettait un bon film d'action science-fictionnel bourré d'affrontements câblés et d'effets visuels Matrixiens. Verdict : c'est sympa, c'est marrant, c'est bien foutu mais c'est hélas aussi très lourd. Tournant en rond au bout d'une vingtaine de minutes, le long-métrage s'avère très répétitif et surtout très peu captivant au final, le scénario n'arrivant jamais à aller plus loin que le bout de son nez. Quant au casting, c'est également peu convaincant... Jet Li peine à convaincre dans son double rôle de gentil paumé et de méchant diabolique tandis que son comparse de fortune Jason Statham (Ghosts of Mars) se régale à faire le big boy qui flingue tout sur son passage. S'ajoutent également au casting le polyvalent Delroy Lindo et la sexy Carla Gugino, campant respectivement le traqueur du bad guy et la comparse du héros. The One est donc une série B tout ce qu'il y a de plus banale où scènes d'action filmées au ralenti, effets visuels à outrance et musique rock omniprésente se fracassent constamment dans un gros bordel pas toujours digérable. Et si le film confirme une belle place dans le cinéma d'action pour Jason Statham, il nous fait amèrement regretter le passage Outre-Atlantique d'un Jet Li confiné aux rôles indigestes voire nanardesques."))
print(predire("Film pas mauvais, qui se laisse regarder. Très belle performance de Wahlberg pas habitué à jouer ce genre de rôle et qui réussit son pari."))
print(predire("Un film d'un grand intérêt, pas mauvais du tout."))


0.005876737646758556
0.9777987599372864
0.9778622388839722


| Avis | Vraie réponse | TF-IDF + bigrammes (proba positif) | CamemBERT (proba positif) |
|---|---|---|---|
| Piège 1 (« génial ») | négatif | 22 % | 0,6 % |
| Piège 2 (« pas mauvais ») | positif | 38 % | 97,8 % |
| Piège 3 (inventé) | positif | 0,5 % | 97,8 % |

**Ma conclusion** : CamemBERT réussit les 3 pièges, là où TF-IDF en ratait 2. TF-IDF additionne des poids de mots sans regarder leur ordre : « mauvais » pèse négatif quel que soit son voisin. Grâce à l’attention, chaque mot de CamemBERT est interprété en fonction des autres : « mauvais » précédé de « pas » change de sens, et « génial » est rattaché à un autre film que celui critiqué. Attention cependant : 3 exemples ne prouvent rien à eux seuls ; c’est l’accuracy sur 20 000 avis qui le confirme.

## 3.8 Sauvegarder le modèle dans Google Drive

⚠️ **Tout ce qui est sur Colab disparaît à la fin de la session.** On sauvegarde le modèle dans ton Google Drive pour le réutiliser à l'étape 5 (démo). Colab va te demander l'autorisation d'accéder à ton Drive.

Le modèle pèse environ 450 Mo.

In [51]:
from google.colab import drive

drive.mount("/content/drive")

dossier = "/content/drive/MyDrive/camembert-allocine"
trainer.save_model(dossier)
tokenizer.save_pretrained(dossier)
print("Modèle sauvegardé dans", dossier)

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle sauvegardé dans /content/drive/MyDrive/camembert-allocine
